### Pipeline as components
파이프라인을 component로 사용했을때 output parsing 시 문제 생김</br>
https://github.com/kubeflow/pipelines/issues/10039 -> closed by https://github.com/kubeflow/pipelines/pull/11196 </br>
https://github.com/kubeflow/pipelines/blob/master/CHANGELOG.md#250-2025-04-28 -> 2.5.0 에서 버그 픽스

In [1]:
from kfp import components
from kfp import dsl
import kfp

kfp_client = kfp.Client()

download_dataset_pipeline = components.load_component_from_file('download_dataset_pipeline.yaml')
preparing_model_pipeline = components.load_component_from_file('preparing_model_pipeline.yaml')
create_inference_service_pipeline = components.load_component_from_file('create_inference_service_pipeline.yaml')

/opt/conda/lib/python3.11/site-packages/kfp/client/client.py:159: FutureWarning: This client only works with Kubeflow Pipeline v2.0.0-beta.2 and later versions.
  warnings.warn(


In [2]:
dir(preparing_model_pipeline)
create_inference_service_pipeline.name

'create-inference-service-pipeline'

In [3]:
print(download_dataset_pipeline.required_inputs)
print(preparing_model_pipeline.required_inputs)
print(create_inference_service_pipeline.required_inputs)

['current_sc', 'dataset_pvc_name', 'dataset_pvc_size', 'download_url', 'output_file']
['base_model', 'datasets_pvc_name', 'epochs', 'mlflow_experiment', 'mlflow_s3_url', 'mlflow_url']
['isvc_name', 'model_s3_url', 'namespace', 'num_of_gpus', 'runtime_name', 'secret_name', 'service_account_name']


In [4]:
@dsl.pipeline(
    name="e2e_pipe_lp_model"
)
def end_to_end_pipeline_prepare_models(
    download_url: str,
    output_file: str,
    dataset_pvc_name: str,
    dataset_pvc_size: str,
    current_sc: str,
    base_model: str,
    mlflow_url: str,
    mlflow_s3_url: str,
    mlflow_experiment: str,
    epochs: int,
    namespace: str,
    secret_name: str,
    service_account_name: str,
    runtime_name: str,
    isvc_name:str,
    num_of_gpus:str
):
    task1 = download_dataset_pipeline(
        download_url=download_url,
        output_file=output_file,
        dataset_pvc_name=dataset_pvc_name,
        dataset_pvc_size=dataset_pvc_size,
        current_sc=current_sc
    )
    task2 = preparing_model_pipeline(
        base_model=base_model,
        datasets_pvc_name=task1.output,
        mlflow_url=mlflow_url,
        mlflow_s3_url=mlflow_s3_url,
        mlflow_experiment=mlflow_experiment,
        epochs=epochs
    )
    task3 = create_inference_service_pipeline(
        namespace=namespace,
        secret_name=secret_name,
        service_account_name=service_account_name,
        runtime_name=runtime_name,
        isvc_name=isvc_name,
        model_s3_url=task2.output,
        num_of_gpus=num_of_gpus,
    )

In [5]:
import os

download_url = "https://universe.roboflow.com/ds/SV8XowFrNf?key=2NutaEpgE9" # Enter your url from roboflow
output_file = "/data/roboflow.zip"
dataset_pvc_name = "roboflow-lp-datasets"
dataset_pvc_size = '5Gi'
current_sc = os.popen("kubectl get pvc user-pvc -o=jsonpath='{.spec.storageClassName}'").read()
base_model = "yolo11s"
# datasets_pvc_name = 'roboflow-license-plate-datasets-e2e'
mlflow_url = "https://mlflow.ingress.pcai0308.sg2.hpecolo.net"
mlflow_s3_url = "http://local-s3-service.ezdata-system.svc.cluster.local:30000"
mlflow_experiment = 'license_plate_yolo11s_finetune'
namespace=os.popen("kubectl get pvc user-pvc -o=jsonpath='{.metadata.namespace}'").read()
secret_name = "my-secret"
service_account_name = "my-service-account"
runtime_name = "my-ngc"
isvc_name = "my-isvc"
# model_s3_url='s3://mlflow.sg2pcai172/7/472e4f614ff341abbcac27b5db3be826/artifacts' + '/triton/triton_engines'
num_of_gpus="1"

In [6]:
kfp_client.create_run_from_pipeline_func(
    end_to_end_pipeline_prepare_models,
    arguments={
        'download_url': download_url,
        'output_file': output_file,
        'dataset_pvc_name': dataset_pvc_name,
        'dataset_pvc_size': dataset_pvc_size,
        'current_sc': current_sc,
        'base_model': base_model,
        # 'datasets_pvc_name': datasets_pvc_name,
        'epochs': 1,
        'mlflow_url': mlflow_url,
        'mlflow_s3_url': mlflow_s3_url,
        'mlflow_experiment': mlflow_experiment,
        'namespace': namespace,
        'secret_name': secret_name,
        'service_account_name': service_account_name,
        'runtime_name': runtime_name,
        'isvc_name': isvc_name,
        # 'model_s3_url': model_s3_url,
        'num_of_gpus': num_of_gpus
    },
    experiment_name="test-rhgt-exp",
)

RunPipelineResult(run_id=cfcc5cad-0094-4bfd-ae5c-ca5bf65f1d31)